# Multi‐Modal Classification Pipeline

## Project Overview

**Objective**  
Develop a unified multi-modal classification and transcription pipeline using the Flickr8k dataset, leveraging image, audio and text cues to categorize each sample and evaluate speech transcription quality.

**Dataset**  
- **Images**: 8,000 JPEGs with up to five captions each  
- **Audio**: Corresponding WAV recordings of spoken captions  
- **Captions**: Textual descriptions aligned with image/audio pairs  
- **Classes**: Four categories (people, animals, vehicles, other) assigned via keyword heuristics

**Outcome**  
A scalable, modular framework that demonstrates the relative performance of individual modalities versus their fusion, and provides quantitative transcription quality metrics.  


## 1. Dependencies
## Imports & Environment Setup

This section loads all required libraries and configures the compute device.

1. **Operating System & Paths**  
   ```python
   import os  
   from pathlib import Path  
   Manage files, directories, and platform-agnostic paths.

2. **Data Processing**  
   ```python
   import numpy as np  
   import pandas as pd  
   Numeric arrays (NumPy) and tabular data (Pandas).

3. **Media I/O**  
   ```python
   from PIL import Image  
   Load and convert images.  
   import soundfile as sf  
   Read/write WAV audio files.

4. **Progress Bars**
   ```python  
   from tqdm import tqdm  
   Command-line progress bar.  
   from tqdm.notebook import tqdm  
   Jupyter-friendly progress bar.

5. **Deep Learning (PyTorch)**
   ```python  
   import torch  
   import torch.nn.functional as F
   from torch import nn, optim 
   from torch.utils.data import Dataset, DataLoader  
   Core framework, layers, loss/functions, optimizers, data pipelines.

6. **Pretrained Transformers**  
   ```python
   from transformers import (
       ViTImageProcessor, ViTModel,           # Vision Transformer
       Wav2Vec2Processor, Wav2Vec2Model,       # Audio feature extractor
       BertTokenizer, BertModel,               # Text encoder (BERT)
       WhisperProcessor, WhisperForConditionalGeneration  # Speech transcription
   )


In [1]:
import os # Operating system interfaces
import numpy as np # Numerical computing library
import pandas as pd # Data analysis and manipulation
from pathlib import Path # Object-oriented filesystem paths
from PIL import Image # Image loading and processing
from tqdm import tqdm # CLI progress bar
import soundfile as sf # Audio I/O (read/write WAV files)

import torch # Core PyTorch library
import torch.nn.functional as F # Neural network functions (losses, activations)
from torch import nn, optim # Neural network modules and optimizers
from torch.utils.data import Dataset, DataLoader # Dataset abstraction and batching
from tqdm.notebook import tqdm  # Jupyter progress bar

from transformers import (  # Pretrained model processors and classes
    ViTImageProcessor, ViTModel, # Vision Transformer (image features)
    Wav2Vec2Processor, Wav2Vec2Model, # Wav2Vec2 (audio features)
    BertTokenizer, BertModel, # BERT (text features)
    WhisperProcessor, WhisperForConditionalGeneration  # Whisper (speech transcription)
)
from sklearn.model_selection import train_test_split  # Data splitting utility
from sklearn.metrics import accuracy_score  # Classification accuracy metric
import evaluate # Evaluation metrics (BLEU, ROUGE, etc.)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Select GPU if available
print(f"Using device: {device}")  # Display the compute device in use


Using device: cuda


## 2. Build Unified DataFrame & Train/Test Split

This cell performs the following steps:

1. **Parse Captions**  
   - Reads `Flickr8k.token.txt`, splitting each line into an image identifier (`image_id`), caption index (`cap_idx`) and the caption text.  
   - Stores these records in `df_caps`.

2. **Collect Image Paths**  
   - Scans the `images/` folder for `.jpg` files.  
   - Creates `df_imgs` mapping each `image_id` to its file path.

3. **Collect Audio Paths**  
   - Scans `flickr8k_audio/wavs/` for `.wav` files.  
   - Extracts `image_id` and `cap_idx` from filenames (e.g. `12345_2.wav`).  
   - Creates `df_auds` mapping each caption to its audio file.

4. **Merge DataFrames**  
   - Joins `df_caps`, `df_imgs` and `df_auds` on `image_id` and `cap_idx`.  
   - Produces a unified `df` with columns:  
     `image_id`, `cap_idx`, `caption`, `img_path`, `audio_path`.

5. **Assign Class Labels**  
   - Defines `get_label(txt)` to map caption keywords to one of four classes:  
     0 = people, 1 = animals, 2 = vehicles, 3 = other.  
   - Applies this function to populate the `label` column.

6. **Stratified Train/Test Split**  
   - Uses `train_test_split(..., test_size=0.1, stratify=df["label"], random_state=42)` to split `df` into:  
     - `train_df` (90% of samples)  
     - `test_df`  (10% of samples)  
   - Resets indices on both DataFrames.  
   - Prints the resulting train/test sizes.


In [ ]:
records = []  # Initialize list to hold caption records
with open("flickr8k_text/Flickr8k.token.txt") as f:  # Open captions file
    for line in f:  # Iterate over each line
        tag, cap = line.strip().split("\t")  # Split into “image#idx” and caption
        img, idx = tag.split("#")  # Separate filename (“.jpg”) from caption index
        img_id = img.replace(".jpg", "")  # Remove “.jpg” extension for ID
        records.append({
            "image_id": img_id,         # Caption’s image identifier
            "cap_idx": int(idx),        # Caption index (int)
            "caption": cap              # Caption text
        })

df_caps = pd.DataFrame(records)  # Create DataFrame of captions

# image paths
df_imgs = pd.DataFrame([
    {
        "image_id": p.stem,           # Image identifier (filename without extension)
        "img_path": str(p)            # Full path to image file
    }
    for p in Path("images").glob("*.jpg")  # All JPEG files in “images” folder
])

# audio paths
aud_recs = []  # Initialize list for audio file records
for p in Path("flickr8k_audio/wavs").glob("*.wav"):  # Iterate through WAV files
    base = p.stem.rsplit("_", 1)  # Split “imageID_capIdx”
    if len(base) == 2:  # Ensure valid split
        aud_recs.append({
            "image_id": base[0],     # Corresponding image ID
            "cap_idx": int(base[1]), # Caption index
            "audio_path": str(p)     # Full path to audio file
        })

df_auds = pd.DataFrame(sorted(
    aud_recs,
    key=lambda r: (r["image_id"], r["cap_idx"])
))  # Sort records and create DataFrame

# merge
df = (
    df_caps
    .merge(df_imgs, on="image_id")             # Merge captions with images
    .merge(df_auds, on=["image_id", "cap_idx"])  # Merge resulting DataFrame with audios
)

# label assignment
def get_label(txt):  # Map caption to class label
    t = txt.lower()  # Case-insensitive matching
    if any(w in t for w in ("man", "woman", "boy", "girl", "person")):
        return 0      # People class
    if any(w in t for w in ("dog", "cat", "horse", "bird")):
        return 1      # Animals class
    if any(w in t for w in ("car", "truck", "bus", "bike", "airplane", "train", "boat")):
        return 2      # Vehicles class
    return 3          # Other / miscellaneous

df["label"] = df["caption"].apply(get_label)  # Apply label assignment

# split
train_df, test_df = train_test_split(
    df,
    test_size=0.1, # 10% for testing
    stratify=df["label"], # Maintain label distribution
    random_state=42 # Ensure reproducibility
)
train_df = train_df.reset_index(drop=True)  # Reset train DataFrame index
test_df  = test_df.reset_index(drop=True)   # Reset test DataFrame index

print(f"Train/test sizes: {len(train_df)}/{len(test_df)}")  # Display split sizes


Train/test sizes: 36000/4000


## 3. Dataset & DataLoader

This section defines how raw samples (images, audio, captions, labels) are loaded from disk and batched for training and evaluation.

1. **`Flickr8kDataset`**  
   - Inherits `torch.utils.data.Dataset`  
   - **`__init__(self, df)`**: stores a Pandas DataFrame containing `img_path`, `audio_path`, `caption` and `label`.  
   - **`__len__(self)`**: returns the total number of samples.  
   - **`__getitem__(self, i)`**:  
     1. Retrieves the i-th row from the DataFrame.  
     2. Loads the image with `PIL.Image.open(...).convert("RGB")`.  
     3. Reads the audio waveform and sample rate with `soundfile.read(...)`.  
     4. Returns a tuple: `(image, waveform, sample_rate, caption, label)`.

2. **`collate_fn(batch)`**  
   - Receives a list of samples (tuples) from `__getitem__`.  
   - Unzips into separate lists for images, audio arrays, sample rates, captions.  
   - Stacks labels into a single 1D `torch.Tensor`.  
   - Ensures each batch has the shape expected by the training loops.

3. **DataLoaders**  
   - **`train_loader`**  
     - Wraps `Flickr8kDataset(train_df)`  
     - `batch_size=4`, `shuffle=True` (randomize each epoch)  
     - Uses `collate_fn` to assemble batches  
   - **`test_loader`**  
     - Wraps `Flickr8kDataset(test_df)`  
     - `batch_size=4`, `shuffle=False` (deterministic evaluation)  
     - Uses the same `collate_fn`  


In [ ]:
from torch.utils.data import Dataset, DataLoader  # Dataset abstraction and batch loader

class Flickr8kDataset(Dataset):
    def __init__(self, df):
        self.df = df   # Store DataFrame with paths, captions, labels

    def __len__(self):
        return len(self.df) # Total number of samples

    def __getitem__(self, i):
        r = self.df.iloc[i]  # Retrieve row at index i
        img = Image.open(r.img_path).convert("RGB") # Load image and ensure RGB format
        wav, sr = sf.read(r.audio_path) # Load audio waveform and sample rate
        return img, wav, sr, r.caption, r.label # Return image, audio, caption, label

def collate_fn(batch):
    imgs, auds, srs, caps, labs = zip(*batch) # Unzip batch elements
    return list(imgs), list(auds), list(srs), list(caps), torch.tensor(labs)  # Return lists and tensor of labels

train_loader = DataLoader(
    Flickr8kDataset(train_df), # Training dataset
    batch_size=4, # Four samples per batch
    shuffle=True, # Shuffle every epoch
    collate_fn=collate_fn  # Custom batch assembler
)

test_loader = DataLoader(
    Flickr8kDataset(test_df), # Test dataset
    batch_size=4, # Four samples per batch
    shuffle=False,  # No shuffling for evaluation
    collate_fn=collate_fn # Same assembler
)


## 4. Feature Extraction

Defines four functions that convert raw inputs into fixed-length feature vectors for downstream classification:

- **Image** (`extract_image_feats`):  
  Uses a pretrained Vision Transformer (ViT-base) to process a batch of PIL images and returns the 768-dimensional CLS-token embeddings.

- **Speech** (`extract_speech_feats`):  
  Feeds padded or truncated audio arrays through the Whisper encoder, then mean-pools its hidden states into a vector of size `d_model`.

- **Text** (`extract_text_feats`):  
  Tokenizes captions with BERT-base, forwards them through the model, and extracts the 768-dimensional CLS-token embeddings.

- **Fusion** (`extract_fusion_feats`):  
  Concatenates the image, speech and text feature vectors along the feature dimension, yielding a combined vector of length `IMG_DIM + d_model + TXT_DIM`.

Note: Speech, text and fusion extractors run under `torch.no_grad()` to disable gradient computation.  


In [ ]:
# Image (ViT)

from transformers import ViTImageProcessor, ViTModel  # ViT image processor & model

vit_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")  # Load ViT processor
vit_model     = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device)  # Load ViT model to device
IMG_DIM       = vit_model.config.hidden_size  # Feature dimension (768)

def extract_image_feats(images, audios, srs, captions):
    """
    images: list of PIL.Image
    returns: Tensor of shape [B, IMG_DIM]
    """
    inputs = vit_processor(images=images, return_tensors="pt").to(device)  # Preprocess images
    outputs = vit_model(**inputs)  # Forward pass through ViT
    return outputs.last_hidden_state[:, 0]  # Return CLS token embeddings [B, IMG_DIM]

# Audio (Whisper encoder)

from transformers import WhisperProcessor, WhisperForConditionalGeneration  # Whisper processor & model

whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-base")  # Load Whisper processor
whisper_model     = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base").to(device)  # Load Whisper model

c1 = whisper_model.model.encoder.conv1.stride[0]  # Conv1 stride value
c2 = whisper_model.model.encoder.conv2.stride[0]  # Conv2 stride value
maxpos = whisper_model.config.max_source_positions  # Max input positions
EXPECTED_LEN = maxpos * c1 * c2  # Expected feature length (e.g. 3000)

import numpy as np  # Numerical operations
import torch.nn.functional as F  # Functional API for tensor ops

@torch.no_grad()
def extract_speech_feats(images, audios, srs, captions):
    """
    audios: list of 1D np.ndarray or torch.Tensor
    srs:    list of sample rates
    returns: Tensor of shape [B, H]
    """
    raw = []  # List to hold float32 numpy arrays
    for a in audios:  # Iterate over audio samples
        arr = a if isinstance(a, np.ndarray) else a.numpy()  # Convert tensor to numpy if needed
        raw.append(arr.astype(np.float32))  # Cast to float32

    batch = whisper_processor(raw, sampling_rate=srs[0], return_tensors="pt", padding=True)  # Preprocess & pad
    feats = batch.input_features.to(device)  # Move to device [B, n_mels, seq_len]

    seq_len = feats.shape[-1]  # Current sequence length
    if seq_len < EXPECTED_LEN:  # If shorter than expected
        feats = F.pad(feats, (0, EXPECTED_LEN - seq_len))  # Pad time dimension
    else:
        feats = feats[..., :EXPECTED_LEN]  # Truncate to expected length

    enc = whisper_model.model.encoder(feats).last_hidden_state  # Encoder output [B, L, H]
    return enc.mean(dim=1)  # Mean-pool over time [B, H]

# Text (BERT)

from transformers import BertTokenizer, BertModel  # BERT tokenizer & model

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # Load BERT tokenizer
bert_model     = BertModel.from_pretrained("bert-base-uncased").to(device)  # Load BERT model to device
TXT_DIM        = bert_model.config.hidden_size  # Feature dimension (768)

@torch.no_grad()
def extract_text_feats(images, audios, srs, captions):
    """
    captions: list of strings
    returns: Tensor of shape [B, TXT_DIM]
    """
    enc = bert_tokenizer(captions, padding=True, truncation=True, return_tensors="pt").to(device)  # Tokenize & batch
    outputs = bert_model(**enc)  # Forward pass through BERT
    return outputs.last_hidden_state[:, 0]  # Return CLS token embeddings [B, TXT_DIM]

#Fusion (concatenate all three)

@torch.no_grad()
def extract_fusion_feats(images, audios, srs, captions):
    """
    returns: Tensor of shape [B, IMG_DIM+H+TXT_DIM]
    """
    fi = extract_image_feats(images, audios, srs, captions)  # Extract image features
    fs = extract_speech_feats(images, audios, srs, captions)  # Extract speech features
    ft = extract_text_feats(images, audios, srs, captions)    # Extract text features
    return torch.cat([fi, fs, ft], dim=1)  # Concatenate features [B, IMG+H+TXT]


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 5. Heads, Loss & Optimizers

This cell configures the classification layers, the loss function, and the optimizers in one place:

- **`NUM_C = 4`**  
  Defines the number of output classes.

- **Classification heads**  
  - `img_head`: linear layer mapping `IMG_DIM` → `NUM_C`  
  - `aud_head`: linear layer mapping `d_model` → `NUM_C`  
  - `txt_head`: linear layer mapping `TXT_DIM` → `NUM_C`  
  - `fus_head`: linear layer mapping `(IMG_DIM + d_model + TXT_DIM)` → `NUM_C`  

- **Loss function**  
  ```python
  criterion = nn.CrossEntropyLoss()


In [ ]:
NUM_C = 4   # Define number of output classes

img_head = nn.Linear(IMG_DIM, NUM_C).to(device)  # Image‐only classifier head
aud_head = nn.Linear(whisper_model.config.d_model, NUM_C).to(device)  # Audio‐only classifier head
txt_head = nn.Linear(TXT_DIM, NUM_C).to(device)   # Text‐only classifier head
fus_head = nn.Linear(  # Fusion classifier head
    IMG_DIM + whisper_model.config.d_model + TXT_DIM,
    NUM_C
).to(device)

criterion = nn.CrossEntropyLoss() # Classification loss function

opt_img = optim.Adam( # Optimizer for ViT + image head
    list(vit_model.parameters()) + list(img_head.parameters()),
    lr=1e-4
)
opt_aud = optim.Adam( # Optimizer for audio head
    aud_head.parameters(),
    lr=1e-4
)
opt_txt = optim.Adam( # Optimizer for text head
    txt_head.parameters(),
    lr=1e-4
)
opt_fus = optim.Adam( # Optimizer for fusion head
    fus_head.parameters(),
    lr=1e-4
)


## 6. Training & Evaluation Loops

This cell defines two core functions for model training and evaluation:

1. **`train_loop(feat_fn, head, optimizer, loader, desc)`**  
   - Sets the classification head to training mode.  
   - Iterates over batches from `loader`, displaying a progress bar labeled by `desc`.  
   - For each batch:
     - Moves labels to the compute device.
     - Extracts features via `feat_fn(images, audios, srs, captions)`.
     - Computes logits with `head(feats)` and loss via `criterion`.
     - Clears previous gradients, backpropagates, and updates weights with `optimizer`.

2. **`eval_loop(feat_fn, head)`**  
   - Sets the head to evaluation mode and disables gradient tracking.  
   - Iterates over `test_loader` batches:
     - Extracts features and obtains predicted class indices.
     - Collects predictions and true labels into lists.
   - Returns the overall accuracy computed by `accuracy_score(labs, preds)`.


In [ ]:
#Update train_loop to take a loader & a desc
def train_loop(feat_fn, head, optimizer, loader, desc):
    head.train()  # enable training mode
    for images, audios, srs, captions, labels in tqdm(loader, desc=desc):# iterate batches with progress bar
        labels = labels.to(device) # move labels to GPU/CPU
        feats  = feat_fn(images, audios, srs, captions) # extract features
        logits = head(feats) # compute class logits
        loss   = criterion(logits, labels) # compute loss

        optimizer.zero_grad()  # clear previous gradients
        loss.backward() # backpropagate
        optimizer.step() # update weights

def eval_loop(feat_fn, head):
    head.eval() # enable eval mode
    preds, labs = [], []  # init prediction and label lists
    with torch.no_grad(): # disable gradient tracking
        for imgs, auds, srs, caps, lb in test_loader:  # iterate test batches
            feats = feat_fn(imgs, auds, srs, caps) # extract features
            p = head(feats).argmax(1).cpu().numpy() # predict class indices
            preds += p.tolist() # collect predictions
            labs  += lb.numpy().tolist() # collect true labels
    return accuracy_score(labs, preds) # return accuracy metric


## 6.1 Image-Only Training & Evaluation

This cell trains the image-only classifier head and then reports its test accuracy.

```python
# Train the image-only model
train_loop(
    extract_image_feats,            # ViT-based image feature extractor
    img_head,                       # Image-only classification head
    opt_img,                        # Optimizer for ViT + img_head
    train_loader,                   # Training DataLoader
    desc="▶ Training Image-only"    # Progress bar label
)

# Evaluate on the test set
print(
    "Image-only Acc:",
    eval_loop(extract_image_feats, img_head)  # Compute and display accuracy
)


In [ ]:
# 6.1 Image-only
# Train only the image classification head
train_loop(
    extract_image_feats, # feature extractor for images
    img_head, # image‐only classifier head
    opt_img, # optimizer for image head (and ViT)
    train_loader, # training data loader
    desc="▶ Training Image‐only" # progress bar description
)

# Evaluate image‐only model on test set
print(
    "Image‐only Acc:", 
    eval_loop(extract_image_feats, img_head)  # compute and display accuracy
)


▶ Training Image‐only:   0%|          | 0/9000 [00:00<?, ?it/s]

Image‐only Acc: 0.72925


## 6.2 Speech-Only Training & Evaluation

This cell trains the audio-only classifier head using Whisper features and then prints its test accuracy.

```python
print("Speech-only:")  # Header

train_loop(
    extract_speech_feats,          # Whisper-based audio feature extractor
    aud_head,                      # Audio-only classification head
    opt_aud,                       # Optimizer for aud_head
    train_loader,                  # Training DataLoader
    desc="▶ Training Speech-only"  # Progress bar label
)

print(
    " → Acc:",
    eval_loop(extract_speech_feats, aud_head)  # Compute and display accuracy
)


In [ ]:
# 6.2 Speech-only
print("Speech-only:")  # indicate start of speech‐only training

train_loop(
    extract_speech_feats, # feature extractor for audio
    aud_head, # audio‐only classifier head
    opt_aud, # optimizer for audio head
    train_loader, # training data loader
    desc="▶ Training Speech-only" # progress bar description
)

print(
    " → Acc:", 
    eval_loop(extract_speech_feats, aud_head) # evaluate and print speech‐only accuracy
)


Speech-only:


▶ Training Speech-only:   0%|          | 0/9000 [00:00<?, ?it/s]

 → Acc: 0.4975


## 6.3 Text-Only Training & Evaluation

This cell trains the text-only classifier head using BERT-derived features and then reports its accuracy on the test set.

```python
print("Text-only:")  # Section header

train_loop(
    extract_text_feats,       # BERT-based feature extractor for captions
    txt_head,                 # Text-only classification head
    opt_txt,                  # Optimizer for the text head
    train_loader,             # Training data loader
    desc="▶ Training Text-only"  # Progress bar label
)

print(
    " → Acc:",
    eval_loop(extract_text_feats, txt_head)  # Compute and display text-only accuracy
)


In [ ]:
# 6.3 Text-only
print("Text-only:")  # Print section header for text‐only training

train_loop(
    extract_text_feats, # Use BERT feature extractor on captions
    txt_head, # Text‐only classification head
    opt_txt, # Optimizer for the text head
    train_loader, # Training DataLoader
    desc="▶ Training Text-only"  # Progress bar label
)

print(
    " → Acc:", 
    eval_loop(extract_text_feats, txt_head)  # Compute and display text‐only accuracy
)


Text-only:


▶ Training Text-only:   0%|          | 0/9000 [00:00<?, ?it/s]

 → Acc: 0.7925


## 6.4 Fusion Training & Evaluation

This cell trains the fused‐features classifier head (image + audio + text) and then reports its test accuracy.

```python
print("Fusion:")  # Section header

train_loop(
    extract_fusion_feats,     # Feature extractor that concatenates image, audio, and text embeddings
    fus_head,                 # Fusion classification head
    opt_fus,                  # Optimizer for the fusion head
    train_loader,             # Training DataLoader
    desc="▶ Training Fusion"  # Progress bar label
)

print(
    " → Acc:",
    eval_loop(extract_fusion_feats, fus_head)  # Compute and display fusion model accuracy
)


In [ ]:
# 6.4 Fusion
print("Fusion:")  # Print section header for the fused model

train_loop(
    extract_fusion_feats, # Feature extractor that concatenates image, audio, text
    fus_head, # Fusion classification head
    opt_fus,  # Optimizer for fusion head
    train_loader,  # Training DataLoader
    desc="▶ Training Fusion"     # Progress bar description
)

print(
    " → Acc:", 
    eval_loop(extract_fusion_feats, fus_head)  # Evaluate fusion model and print accuracy
)


Fusion:


▶ Training Fusion:   0%|          | 0/9000 [00:00<?, ?it/s]

 → Acc: 0.827


## Whisper Speech Processor & Model Setup

This cell loads and configures the OpenAI Whisper components for speech transcription:

1. **Imports**  
   ```python
   from transformers import WhisperProcessor, WhisperForConditionalGeneration


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration  # Import Whisper processor and model classes

# Choose model size: “openai/whisper-small” for smaller footprint or “openai/whisper-base” for higher accuracy
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-base")  # Load Whisper tokenizer & feature extractor
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base").to(device)  # Load Whisper model onto GPU/CPU


## 7. Transcription Quality Evaluation (BLEU & ROUGE)

This cell computes standard NLP metrics on Whisper’s predicted transcriptions versus reference captions:

1. **Load evaluation metrics**  
   ```python
   bleu  = evaluate.load("bleu")   # BLEU score for n-gram precision
   rouge = evaluate.load("rouge")  # ROUGE-L for longest common subsequence


In [15]:
# 7) BLEU & ROUGE on Whisper Transcriptions
bleu  = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# flatten back to sentences
preds_sents = [" ".join(tokens) for tokens in preds]
refs_sents  = [" ".join(ref_list[0]) for ref_list in refs]

b = bleu.compute(predictions=preds_sents, references=refs_sents)["bleu"]
r = rouge.compute(predictions=preds_sents, references=refs_sents)["rougeL"]
print(f"BLEU: {b:.4f}   ROUGE-L: {r:.4f}")


BLEU: 0.8845   ROUGE-L: 0.9541
